## Data Owner 

In [1]:
!uv pip install huggingface-hub transformers torch

Audited 3 packages in 13ms


### Step 1: Setup RDS Server

In [2]:
from syft_rds.orchestra import setup_rds_server, remove_rds_stack_dir

In [3]:
remove_rds_stack_dir(key="enclave")

2025-08-11 17:25:25.058 | INFO     | syft_rds.orchestra:remove_rds_stack_dir:101 - ✅ Successfully removed directory /var/folders/4w/9cvj7hqd4_386stwby6n2pcw0000gn/T/enclave


In [4]:
do_stack_1 = setup_rds_server(email="do1@openmined.org", key="enclave")

2025-08-11 17:25:25.067 | INFO     | syft_rds.orchestra:setup_rds_server:221 - Launching mock RDS server in /private/var/folders/4w/9cvj7hqd4_386stwby6n2pcw0000gn/T/enclave


In [5]:
do_client_1 = do_stack_1.init_session(host="do1@openmined.org")

### Step 2.1: Create Private / Mock Dataset

In [6]:


DATASET_PRIVATE_PATH = f"./gpt2"
DATASET_MOCK_PATH = f"./gpt2_mock"
README_PATH = "./readme.md"


In [7]:
# third party
from huggingface_hub import snapshot_download

MODEL_DIR = "./gpt2"

snapshot_download(
    repo_id="openai-community/gpt2",
    ignore_patterns=[
        "*.tflite",
        "*.msgpack",
        "*.bin",
        "*.ot",
        "*.h5",
        "onnx/*",
    ],
    local_dir=MODEL_DIR,
)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

.gitattributes:   0%|          | 0.00/445 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

'/Users/rasswanths/openmined/syftbox-enclave/notebooks/RDS-Eval/gpt2'

In [ ]:
!rm -rf ./gpt2/README.md ./gpt2/.cache

In [ ]:
# Generate Mock Model Weights
# Comment this out, when using autogenerate_mock=True
MOCK_MODEL_DIR = "./gpt2_mock"

# third party
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

private_model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)
private_model_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
mock_model = AutoModelForCausalLM.from_config(private_model.config_class())
mock_model.save_pretrained(MOCK_MODEL_DIR)
private_model_tokenizer.save_pretrained(MOCK_MODEL_DIR)

### Step 2.2: Load Dataset to SyftBox

In [ ]:
try:
    dataset = do_client_1.dataset.create(
        name="GPT2 Model",
        summary="This Dataset contains the GPT2 Model",
        description_path=README_PATH,
        path=DATASET_PRIVATE_PATH,
        mock_path=DATASET_MOCK_PATH,
    )
    dataset.describe()
except Exception as e:
    print(f"Error: {e}")

In [ ]:
do_client_1.datasets

In [ ]:
dataset = do_client_1.datasets[0]

In [ ]:
dataset.describe()

In [ ]:
dataset.get_mock_path()

## Job Approval

In [ ]:
jobs = do_client_1.jobs.get_all(status="pending_code_review")
jobs

In [ ]:
do_client_1.jobs.approve(job=jobs[0])

In [ ]:
jobs = do_client_1.jobs.get_all(status="approved")
jobs

In [ ]:
# job = do_client_1.jobs[-1]
# do_client_1.jobs.approve(job)